# Your First Real Model: Economic Dispatch

The quick-start showed how a coordinate turns a scalar variable into a vector of decisions, and how `.sum()` and broadcasting line up naturally. Real models usually go a step further: more than one dimension and real input data flowing in from `pandas` or `xarray`.

Let's put both into practice with a small **economic dispatch** problem: decide how much each of three generators should produce in each hour of a day so that demand is met at minimum cost.

## The Problem

Three generators — `coal`, `gas`, and `wind` — must cover an hourly demand profile over 24 hours. Each generator has a fixed capacity and marginal cost. We want to find the cheapest schedule.

Formally, with generators indexed by $g$ and hours by $t$:

$$
\begin{aligned}
\min_{p} \quad & \sum_{g,t} c_g \, p_{g,t} \\
\text{s.t.} \quad & \sum_g p_{g,t} = d_t & \forall t \\
& 0 \le p_{g,t} \le \bar p_g & \forall g, t
\end{aligned}
$$

In [ ]:
import pandas as pd

import linopy

## The data

Real input data usually arrives as `pandas` objects. linopy consumes them directly: each named index becomes a coordinate on the resulting variable.

In [ ]:
generators = pd.DataFrame(
    {
        "capacity": [200, 100, 80],  # MW
        "marginal_cost": [30.0, 60.0, 2.0],  # $/MWh
    },
    index=pd.Index(["coal", "gas", "wind"], name="generator"),
)
generators

In [ ]:
hours = pd.RangeIndex(24, name="hour")
demand = pd.Series(
    [
        200,
        200,
        200,
        200,
        210,
        230,
        260,
        290,
        310,
        320,
        330,
        340,
        350,
        350,
        350,
        340,
        330,
        310,
        290,
        270,
        250,
        230,
        220,
        210,
    ],
    index=hours,
    name="demand",
)
demand.plot(title="Hourly demand (MW)", figsize=(6, 2.5));

## Building the model

The decision variable `p` represents the generation of each unit in each hour. We declare it with two coordinates — `generator` and `hour` — so it has shape `(3, 24)`.

In [ ]:
m = linopy.Model()

p = m.add_variables(
    lower=0,
    coords=[generators.index, hours],
    name="generation",
)
p

## Constraints

**Demand balance.** In every hour, the sum of all generation must equal demand. `p.sum("generator")` collapses `p` along the `generator` dimension, leaving an expression with shape `(hour,)` — which lines up with `demand` automatically.

In [ ]:
m.add_constraints(
    p.sum("generator") == demand,
    name="demand_balance",
)

**Capacity.** Generation of each unit cannot exceed its capacity. The capacity is a `(generator,)`-shaped Series; linopy broadcasts it across the `hour` dimension of `p` automatically.

In [ ]:
m.add_constraints(
    p <= generators["capacity"].to_xarray(),
    name="capacity",
)

## Objective

Minimize total cost: multiply each unit's generation by its marginal cost, then sum across both dimensions.

In [ ]:
m.add_objective(
    (p * generators["marginal_cost"].to_xarray()).sum(),
    sense="min",
)

## Solve

In [ ]:
m.solve()

## Inspecting the solution

The solution is stored on the variable as an xarray `DataArray`, with the same coordinates we declared. Convert it to a `DataFrame` for a tabular view, or plot it directly.

In [ ]:
p.solution.to_pandas().T.head()

In [ ]:
ax = p.solution.to_pandas().T.plot.area(
    figsize=(8, 3.5),
    title="Hourly dispatch (MW)",
)
demand.plot(ax=ax, color="black", linewidth=1.5, label="demand")
ax.set_ylabel("MW")
ax.legend();

## Recap

A few things to notice in the code above:

- The decision variable `p` is 2-dimensional (`generator × hour`). linopy carries those labels everywhere, including in the solution.
- `p.sum("generator")` collapses along the named dimension to produce an expression with shape `(hour,)`, which lines up with `demand` automatically.
- The capacity constraint `p <= capacity` broadcasts a 1-D `(generator,)` array across the 2-D `(generator, hour)` variable — no Python loops, no manual indexing.
- The cheapest generator (`wind`, at 2 $/MWh) runs flat out, `coal` covers most of the rest, and `gas` only kicks in at peak hours.

The same patterns scale up: add more generators, longer horizons, additional dimensions like regions or scenarios — the model code does not change shape.